In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime
from dateutil.relativedelta import relativedelta
import matplotlib.dates as mdates
import seaborn as sns
from FinMind.data import DataLoader
api = DataLoader()

class AdvancedDCAStrategy:
    def __init__(self, base_monthly_investment=10000, fee_min=1, fee_rate=0.001425, dip_config=None):
        self.base_monthly = base_monthly_investment
        self.fee_min = fee_min
        self.fee_rate = fee_rate
        self.dip_config = dip_config or {
            'small':  {'threshold': -0.03, 'multiplier': 1.0,  'active': True},
            'medium': {'threshold': -0.05, 'multiplier': 1.5,  'active': True},
            'big':    {'threshold': -0.10, 'multiplier': 10.0, 'active': True}
        }
        self.reset_state()

    def reset_state(self):
        self.shares_owned = 0
        self.total_invested = 0
        self.total_fees = 0
        self.investment_log = []
        self.dividend_log = []
        self.last_buy_price = 0.0
        self.reference_price = 0.0

    def _calculate_fee(self, amount, is_dca_type=True):
        """
        區分兩種手續費邏輯：
        1. 定期定額 (DCA)：金額 > 10000 收 2 元，否則收 1 元
        2. 跌幅加碼 (Dip)：max(fee_min, amount * fee_rate)
        """
        if is_dca_type:
            return 2 if amount > 10000 else 1
        else:
            return max(self.fee_min, int(amount * self.fee_rate))

    def _execute_buy(self, date, price, amount, reason, is_dca_type):
        """執行買入，區分費用類型並更新基準價"""
        fee = self._calculate_fee(amount, is_dca_type=is_dca_type)
        
        # 實務邏輯：投入總額 = 買股錢 + 手續費
        # 實際能買到的股數 = (投入總額 - 手續費) / 股價
        shares_bought = (amount - fee) / price 
        
        self.shares_owned += shares_bought
        self.total_invested += amount
        self.total_fees += fee
        self.last_buy_price = price  # 更新基準價，供下次加碼判斷使用
        
        self.investment_log.append({
            'date': date,
            'price': price,
            'amount': amount,
            'fee': fee,
            'shares': shares_bought,
            'trigger': reason,
            'total_shares': self.shares_owned
        })

    def _get_trailing_dip_action(self, current_price):
        """核心修正：改為跟蹤最高點後的跌幅"""
        if self.highest_price_since_last_buy == 0:
            return None, 0
            
        # 更新最高點 (如果現價更高，則基準點往上移)
        if current_price > self.highest_price_since_last_buy:
            self.highest_price_since_last_buy = current_price
            return None, 0 # 創新高中，不買入
            
        # 計算從最高點回落的比例
        ratio_from_peak = (current_price - self.highest_price_since_last_buy) / self.highest_price_since_last_buy
        
        configs = sorted(
            [(k, v) for k, v in self.dip_config.items() if v.get('active', False)],
            key=lambda x: x[1]['threshold']
        )
        
        for name, conf in configs:
            if ratio_from_peak <= conf['threshold']:
                return f"{name}從高點回落({conf['threshold']*100}%)", self.base_monthly * conf['multiplier']
        return None, 0

    def _handle_dividends(self, current_date, div_df):
        if not div_df.empty:
            day_div = div_df[div_df['CashExDividendTradingDate'] == current_date]
            if not day_div.empty and self.shares_owned > 0:
                cash_div = day_div['CashEarningsDistribution'].iloc[0]
                income = self.shares_owned * cash_div
                self.dividend_log.append({
                    'date': current_date,
                    'dividend_per_share': cash_div,
                    'dividend_income': income,
                    'shares_owned': self.shares_owned
                })

    def _get_dip_action(self, current_price):
        """檢查跌幅，回傳 (理由, 金額)，若無觸發回傳 (None, 0)"""
        if self.last_buy_price is None:
            return None, 0
            
        ratio = (current_price - self.last_buy_price) / self.last_buy_price
        
        # 依跌幅深度排序，優先觸發大跌 (Big -> Medium -> Small)
        configs = sorted(
            [(k, v) for k, v in self.dip_config.items() if v.get('active', False)],
            key=lambda x: x[1]['threshold']
        )
        
        for name, conf in configs:
            if ratio <= conf['threshold']:
                return f"{name}跌加碼({conf['threshold']*100}%)", self.base_monthly * conf['multiplier']
        return None, 0

    # ---------------------------------------------------------
    # 方式一：定期定額 + 跌幅加碼
    # ---------------------------------------------------------
    def apply_dca_plus_dip(self, df, div_df, invest_day=15):
        self.reset_state()
        df['date'] = pd.to_datetime(df['date'])
        monthly_dates = self._get_actual_trading_dates(df, invest_day)

        for _, row in df.iterrows():
            curr_date, curr_price = row['date'], row['close']
            self._handle_dividends(curr_date, div_df)

            # 1. 優先判斷是否為「定期定額日」
            if curr_date in monthly_dates:
                # 定期定額日，強制執行買入，使用 DCA 費用邏輯
                self._execute_buy(curr_date, curr_price, self.base_monthly, "定期定額", is_dca_type=True)
            
            # 2. 非定期定額日，判斷是否觸發「跌幅加碼」
            else:
                reason, amount = self._get_dip_action(curr_price)
                if reason:
                    # 加碼行為，使用 Dip 費用邏輯
                    self._execute_buy(curr_date, curr_price, amount, reason, is_dca_type=False)

        return pd.DataFrame(self.investment_log), pd.DataFrame(self.dividend_log)

    # ---------------------------------------------------------
    # 方式二：純當日跌幅加碼 (不定期定額，無首日買入)
    # ---------------------------------------------------------
    def apply_only_daily_dip(self, df, div_df):
        """
        修正：移除首日買入，純粹見跌才買。
        """
        self.reset_state()
        df = df.sort_values('date').copy()
        df['date'] = pd.to_datetime(df['date'])
        df['prev_close'] = df['close'].shift(1)

        for _, row in df.iterrows():
            curr_date, curr_price = row['date'], row['close']
            self._handle_dividends(curr_date, div_df)

            # 只要有前一日價格，就判斷跌幅
            if pd.notnull(row['prev_close']):
                daily_change = (curr_price - row['prev_close']) / row['prev_close']
                
                # 遍歷加碼設定
                for level in ['big', 'medium', 'small']:
                    conf = self.dip_config[level]
                    if conf['active'] and daily_change <= conf['threshold']:
                        # 此處視為單次交易，採用非 DCA 費用邏輯
                        self._execute_buy(curr_date, curr_price, self.base_monthly * conf['multiplier'], f"當日跌幅({conf['threshold']*100}%)", is_dca_type=False)
                        break

        return pd.DataFrame(self.investment_log), pd.DataFrame(self.dividend_log)

    def _get_actual_trading_dates(self, df, day):
        df_temp = df.copy()
        df_temp['ym'] = df_temp['date'].dt.to_period('M')
        dates = []
        for ym in df_temp['ym'].unique():
            target = pd.Timestamp(year=ym.year, month=ym.month, day=1) + pd.offsets.Day(day - 1)
            trading_day = df_temp[df_temp['date'] >= target]['date'].min()
            if pd.notnull(trading_day) and trading_day.month == ym.month:
                dates.append(trading_day)
        return pd.DatetimeIndex(dates)
  
    def get_performance(self, current_price):
          """
          计算绩效，包含现金股利收益。
          
          Returns:
          --------
          tuple: (current_value, total_return, return_rate, total_dividend)
          """
          if self.shares_owned == 0:
              return 0, 0, 0, 0
          
          # 计算股票市值
          current_value = self.shares_owned * current_price
          
          # 计算累计现金股利总额
          total_dividend = sum([log['dividend_income'] for log in self.dividend_log]) if hasattr(self, 'dividend_log') else 0
          
          # 总回报 = 当前股票市值 + 累计收到的现金股利 - 总投入本金 + 手續費
          total_return = (current_value + total_dividend) - (self.total_invested + self.total_fees)
          
          # 计算总回报率
          if self.total_invested > 0:
              return_rate = (total_return / self.total_invested) * 100
          else:
              return_rate = 0
          
          return current_value, total_return, return_rate, total_dividend

In [ ]:
today = datetime.today() 
# today = datetime.strptime('2025-04-09', "%Y-%m-%d")
start_date = (today - relativedelta(years=2)).strftime("%Y-%m-%d")
end_date = today.strftime("%Y-%m-%d")
print('start date', start_date)
print('end date', end_date)

ticker = '006208'
df = api.taiwan_stock_daily(
    stock_id=ticker,
    start_date=start_date,
    end_date=end_date
)

div_df = api.taiwan_stock_dividend(
    stock_id=ticker,
    start_date=start_date,
    end_date=end_date
)
div_df = div_df[['CashExDividendTradingDate', 'CashEarningsDistribution']]

2026-03-04 22:28:05.585 | INFO     | FinMind.data.finmind_api:get_data:158 - download Dataset.TaiwanStockPrice, data_id: 006208


start date 2024-03-04
end date 2026-03-04


2026-03-04 22:28:06.359 | INFO     | FinMind.data.finmind_api:get_data:158 - download Dataset.TaiwanStockDividend, data_id: 006208


In [ ]:
cfg = {
    'small':  {'threshold': -0.03, 'multiplier': 2.0,  'active': False},
    'medium': {'threshold': -0.05, 'multiplier': 10,  'active': True},
    'big':    {'threshold': -0.10, 'multiplier': 10.0, 'active': False}
}

strategy = AdvancedDCAStrategy(base_monthly_investment=10000, dip_config=cfg)
pd.set_option('display.max_rows', None)

# 模式一
inv_df1, div_df1 = strategy.apply_dca_plus_dip(df, div_df, invest_day=3)

# 模式二
# inv_df2, div_df2 = strategy.apply_only_daily_dip(df, div_df)

In [ ]:
current_price = df['close'].iloc[-1]
current_value, total_return, return_rate, total_dividend = strategy.get_performance(current_price)

print(f"總投資金額: {strategy.total_invested:,.0f}元")
print(f"總手續費: {strategy.total_fees:,.0f} 元")
print(f"累计现金股利: {total_dividend:.2f} 元")
print(f"目前持有市值: {current_value:,.0f}元")
print(f"總報酬: {total_return:,.0f}元")
print(f"報酬率: {return_rate:.2f}%")
print(f"持有股數: {strategy.shares_owned:.2f}股")

總投資金額: 490,000元
總手續費: 361 元
累计现金股利: 0.00 元
目前持有市值: 776,088元
總報酬: 285,727元
報酬率: 58.31%
持有股數: 4433.52股
